In [15]:
import os
import mlflow
from mlflow.models import ModelSignature
from mlflow.types import Schema, ColSpec

In [ ]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

from dotenv import load_dotenv
load_dotenv()

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

In [18]:
pip_requirements="requirements.txt"

input_schema = Schema([ColSpec("double", "feature1"), ColSpec("double", "feature2")])
output_schema = Schema([ColSpec("double", "prediction")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)
input_example = {"feature1": 0.5, "feature2": 1.5}
metadata = {"model_version": "1.0", "description": "Churn prediction model", "author": "Nikolai Stepanov"}

In [19]:
import os
import pandas as pd
import mlflow
from catboost import CatBoostClassifier  # Import the correct library
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv
import joblib
load_dotenv()

pip_requirements="requirements.txt"
csv_file = "data/initial_data.csv"
df = pd.read_csv(csv_file)

df['target'] = (df['end_date'] != 'No').astype(int) # логика функции
df['end_date'].replace({'No': None}, inplace=True)

pipeline = joblib.load('models/fitted_model.pkl')
data = pd.read_csv('data/initial_data.csv')
data.value_counts('target')

# Fill missing values in data
for col in data.columns:
    if data[col].dtype == 'object':
        data[col] = data[col].fillna('Unknown')
    else:
        data[col] = data[col].fillna(data[col].mean())
        
data.value_counts('target')


/home/mle-user/mle_projects/mle-mlflow/.venv_mlflow/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.4.0 when using version 1.4.1.post1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/mle-user/mle_projects/mle-mlflow/.venv_mlflow/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.0 when using version 1.4.1.post1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/mle-user/mle_projects/mle-mlflow/.venv_mlflow/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersion

target
0    5163
1    1869
Name: count, dtype: int64

In [20]:
pipeline.steps

[('preprocessor',
  ColumnTransformer(transformers=[('binary', OneHotEncoder(drop='if_binary'),
                                   ['paperless_billing', 'internet_service',
                                    'online_security', 'online_backup',
                                    'device_protection', 'tech_support',
                                    'streaming_tv', 'streaming_movies', 'gender',
                                    'partner', 'dependents', 'multiple_lines']),
                                  ('cat', CatBoostEncoder(return_df=False),
                                   ['begin_date', 'end_date', 'type',
                                    'payment_method']),
                                  ('num', StandardScaler(),
                                   ['monthly_charges', 'total_charges'])],
                    verbose_feature_names_out=False)),
 ('model', <catboost.core.CatBoostClassifier at 0x7f35a5921750>)]

In [21]:
pipeline['model']

In [23]:
data_ndarray = pipeline['preprocessor'].fit_transform(data,data['target'])

In [ ]:
# Разделение данных на обучающую и тестовую выборки
train_array, test_array = train_test_split(data_ndarray, test_size=0.2, random_state=42)

# Разделение на признаки (X) и целевую переменную (y)
X_train = train_array[:, :-1]  # Все колонки, кроме последней
y_train = train_array[:, -1]    # Последняя колонка

X_test = test_array[:, :-1]     # Все колонки, кроме последней
y_test = test_array[:, -1]       # Последняя колонка


# pipeline.fit(X_train, y_train)
prediction = y_test
print(X_test,prediction)

signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {'model_type': 'monthly'}
print(signature)



        

[[ 0.          0.          1.         ...  0.10866373  0.19407777
   0.85101491]
 [ 1.          0.          0.         ...  0.42718545  0.19230547
  -1.51736668]
 [ 1.          0.          1.         ...  0.4261478   0.1502369
   1.16846114]
 ...
 [ 0.          0.          1.         ...  0.42851711  0.45151136
   1.1501789 ]
 [ 1.          0.          1.         ...  0.42581885  0.1862973
   0.19617818]
 [ 1.          1.          0.         ...  0.43311728  0.17943967
  -1.32789615]] [ 0.23707189 -0.7966524   0.65476612 ...  1.20960348 -0.97617158
 -0.81553522]
inputs: 
  [Tensor('float64', (-1, 31))]
outputs: 
  [Tensor('float64', (-1,))]
params: 
  None



In [ ]:
model = pipeline['model']

In [29]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

In [32]:
import json

metrics_dict = json.load(open('cv_results/cv_res.json'))

metrics_dict

{'fit_time': 8.905, 'score_time': 0.123, 'test_f1': 1.0, 'test_roc_auc': 1.0}

In [34]:
EXPERIMENT_NAME = "churn_volkovandrey_test"
RUN_NAME = "model_0_registry"
REGISTRY_MODEL_NAME = "churn_model_volkovandrey_test"

os.environ['MLFLOW_S3_ENDPOINT_URL'] = 'https://storage.yandexcloud.net'
os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = os.getenv('AWS_SECRET_ACCESS_KEY')

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id



with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    model_info = mlflow.catboost.log_model( 
        cb_model=model, 
        artifact_path='models',
        await_registration_for=20,
        registered_model_name=REGISTRY_MODEL_NAME, 
        signature=signature,
        input_example=input_example, 
        pip_requirements=pip_requirements,
        metadata=metadata
    )
    mlflow.log_metrics(metrics_dict)

Registered model 'churn_model_volkovandrey_test' already exists. Creating a new version of this model...
2024/11/05 21:12:08 INFO mlflow.tracking._model_registry.client: Waiting up to 20 seconds for model version to finish creation. Model name: churn_model_volkovandrey_test, version 4
Created version '4' of model 'churn_model_volkovandrey_test'.


In [36]:
client = mlflow.MlflowClient()

# Получение информации о моделях
models = client.search_model_versions(filter_string=f"name = '{REGISTRY_MODEL_NAME}'")
print(f"Model info:\n {models}")

model_name_1 = models[-1].name 
model_version_1 = models[-1].version  # Последняя версия
model_stage_1 = models[-1].current_stage  # Статус последней модели

model_name_2 = models[-2].name  # Предпоследняя модель
model_version_2 = models[-2].version  # Предпоследняя версия
model_stage_2 = models[-2].current_stage  # Статус предпоследней модели


print(f"Текущий stage модели 1: {model_stage_1}")
print(f"Текущий stage модели 2: {model_stage_2}")

# поменяйте статус каждой модели
client.transition_model_version_stage(model_name_1, model_version_1, 'production')

client.transition_model_version_stage(model_name_2, model_version_2, 'staging')

# переимнуйте модель в реестре
client.rename_registered_model(name=REGISTRY_MODEL_NAME, new_name=f'{REGISTRY_MODEL_NAME}_b2c')

Model info:
 []


IndexError: list index out of range